# CNN: Optimización y Transfer Learning

En este notebook trabajamos con:

- Una CNN simple en MNIST.
- Una CNN más profunda y optimizada en CIFAR-10.
- Transfer learning con ResNet50 sobre CIFAR-10.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report



## Sección 1 — CNN simple en MNIST


In [2]:
# Cargar dataset MNIST
(x_train_m, y_train_m), (x_test_m, y_test_m) = keras.datasets.mnist.load_data()

# Normalizar a [0, 1]
x_train_m = x_train_m.astype('float32') / 255.0
x_test_m  = x_test_m.astype('float32') / 255.0

# Añadir canal (grayscale)
x_train_m = x_train_m[..., None]
x_test_m  = x_test_m[..., None]

input_shape_m = (28, 28, 1)
num_classes_m = 10

print('MNIST shapes:')
print('x_train:', x_train_m.shape, 'y_train:', y_train_m.shape)
print('x_test :', x_test_m.shape, 'y_test :', y_test_m.shape)



MNIST shapes:
x_train: (60000, 28, 28, 1) y_train: (60000,)
x_test : (10000, 28, 28, 1) y_test : (10000,)


In [3]:
def make_mnist_cnn():
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=input_shape_m),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes_m, activation='softmax'),
    ])
    return model

model_mnist = make_mnist_cnn()
model_mnist.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_mnist = model_mnist.fit(
    x_train_m, y_train_m,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1,
)



d:\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.9441 - loss: 0.1836 - val_accuracy: 0.9823 - val_loss: 0.0612
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9833 - loss: 0.0555 - val_accuracy: 0.9887 - val_loss: 0.0437
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.9879 - loss: 0.0401 - val_accuracy: 0.9892 - val_loss: 0.0389
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.9908 - loss: 0.0301 - val_accuracy: 0.9903 - val_loss: 0.0340
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9921 - loss: 0.0230 - val_accuracy: 0.9892 - val_loss: 0.0405


In [4]:
# Evaluación avanzada en test (Precision, Recall, F1)
y_pred_probs_m = model_mnist.predict(x_test_m)
y_pred_m = np.argmax(y_pred_probs_m, axis=1)

print(classification_report(y_test_m, y_pred_m, digits=4))



313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

           0     0.9969    0.9847    0.9908       980
           1     0.9912    0.9956    0.9934      1135
           2     0.9903    0.9932    0.9918      1032
           3     0.9833    0.9941    0.9887      1010
           4     0.9898    0.9857    0.9878       982
           5     0.9812    0.9922    0.9866       892
           6     0.9916    0.9896    0.9906       958
           7     0.9864    0.9893    0.9879      1028
           8     0.9969    0.9754    0.9860       974
           9     0.9754    0.9822    0.9788      1009

    accuracy                         0.9883     10000
   macro avg     0.9883    0.9882    0.9882     10000
weighted avg     0.9883    0.9883    0.9883     10000



## Sección 2 — CNN más profunda y optimizada en CIFAR-10


In [5]:
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalizar
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

y_train = y_train.reshape(-1)
y_test  = y_test.reshape(-1)

from sklearn.model_selection import train_test_split

x_train_c, x_val_c, y_train_c, y_val_c = train_test_split(
    x_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

input_shape = (32, 32, 3)
num_classes = 10

print('CIFAR-10 shapes:')
print('x_train:', x_train_c.shape, 'y_train:', y_train_c.shape)
print('x_val  :', x_val_c.shape, 'y_val  :', y_val_c.shape)
print('x_test :', x_test.shape,  'y_test :', y_test.shape)



d:\.venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


CIFAR-10 shapes:
x_train: (45000, 32, 32, 3) y_train: (45000,)
x_val  : (5000, 32, 32, 3) y_val  : (5000,)
x_test : (10000, 32, 32, 3) y_test : (10000,)


In [6]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])



In [7]:
def make_cifar_vgg_like():
    inputs = keras.Input(shape=input_shape)

    x = data_augmentation(inputs)

    # Bloque 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Bloque 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Bloque 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

model_cifar_opt = make_cifar_vgg_like()
model_cifar_opt.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [8]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
]

history_cifar_opt = model_cifar_opt.fit(
    x_train_c, y_train_c,
    epochs=50,
    batch_size=64,
    validation_data=(x_val_c, y_val_c),
    callbacks=callbacks,
    verbose=1,
)



Epoch 1/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 43s 59ms/step - accuracy: 0.3238 - loss: 1.8201 - val_accuracy: 0.4378 - val_loss: 1.5338 - learning_rate: 0.0010
Epoch 2/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.4546 - loss: 1.4950 - val_accuracy: 0.5114 - val_loss: 1.3336 - learning_rate: 0.0010
Epoch 3/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 59ms/step - accuracy: 0.5100 - loss: 1.3613 - val_accuracy: 0.5640 - val_loss: 1.2250 - learning_rate: 0.0010
Epoch 4/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 59ms/step - accuracy: 0.5431 - loss: 1.2740 - val_accuracy: 0.6112 - val_loss: 1.0856 - learning_rate: 0.0010
Epoch 5/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 59ms/step - accuracy: 0.5725 - loss: 1.2079 - val_accuracy: 0.6324 - val_loss: 1.0216 - learning_rate: 0.0010
Epoch 6/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.5894 - loss: 1.1633 - val_accuracy: 0.6358 - val_loss: 1.0154 - learning_rate: 0.0010
Epoch 7/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 59ms/step - accuracy: 0.6008 - l

In [9]:
y_val_pred_probs = model_cifar_opt.predict(x_val_c)
y_val_pred = np.argmax(y_val_pred_probs, axis=1)

print('CNN VGG-like optimizada (validación):')
print(classification_report(y_val_c, y_val_pred, digits=4))

y_test_pred_probs = model_cifar_opt.predict(x_test)
y_test_pred = np.argmax(y_test_pred_probs, axis=1)

print('CNN VGG-like optimizada (test):')
print(classification_report(y_test, y_test_pred, digits=4))



157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
CNN VGG-like optimizada (validación):
              precision    recall  f1-score   support

           0     0.8045    0.7820    0.7931       500
           1     0.8659    0.9040    0.8845       500
           2     0.8018    0.5260    0.6353       500
           3     0.5963    0.4460    0.5103       500
           4     0.7291    0.6620    0.6939       500
           5     0.7513    0.5800    0.6546       500
           6     0.5925    0.9160    0.7196       500
           7     0.7343    0.8180    0.7739       500
           8     0.8839    0.8980    0.8909       500
           9     0.7516    0.9200    0.8273       500

    accuracy                         0.7452      5000
   macro avg     0.7511    0.7452    0.7383      5000
weighted avg     0.7511    0.7452    0.7383      5000

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step
CNN VGG-like optimizada (test):
              precision    recall  f1-score   support

           0     0.7980    0.786

## Sección 3 — Transfer learning con ResNet50 en CIFAR-10


In [10]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

base_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)
base_resnet.trainable = False

inputs_tl = keras.Input(shape=input_shape)
x = layers.Resizing(224, 224)(inputs_tl)
x = preprocess_input(x)
x = base_resnet(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs_tl = layers.Dense(num_classes, activation='softmax')(x)

model_tl = keras.Model(inputs_tl, outputs_tl)
model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [11]:
history_tl = model_tl.fit(
    x_train_c, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(x_val_c, y_val_c),
    verbose=1,
)

y_val_pred_probs_tl = model_tl.predict(x_val_c)
y_val_pred_tl = np.argmax(y_val_pred_probs_tl, axis=1)

print('Transfer learning ResNet50 (validación, solo cabeza):')
print(classification_report(y_val_c, y_val_pred_tl, digits=4))



Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1335s 2s/step - accuracy: 0.0984 - loss: 2.3088 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1339s 2s/step - accuracy: 0.0970 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1434s 2s/step - accuracy: 0.0981 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1491s 2s/step - accuracy: 0.0989 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1454s 2s/step - accuracy: 0.0980 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1486s 2s/step - accuracy: 0.0980 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1468s 2s/step - accuracy: 0.0979 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1445s 2s/step - accuracy: 0.0996 - loss: 2.3027 - 

d:\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
# Fine-tuning de las últimas capas de ResNet50
base_resnet.trainable = True
for layer in base_resnet.layers[:-20]:
    layer.trainable = False

model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_tl_ft = model_tl.fit(
    x_train_c, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(x_val_c, y_val_c),
    verbose=1,
)

y_val_pred_probs_tl_ft = model_tl.predict(x_val_c)
y_val_pred_tl_ft = np.argmax(y_val_pred_probs_tl_ft, axis=1)

print('Transfer learning ResNet50 (validación, fine-tuning):')
print(classification_report(y_val_c, y_val_pred_tl_ft, digits=4))



Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1615s 2s/step - accuracy: 0.3025 - loss: 1.9417 - val_accuracy: 0.4472 - val_loss: 1.6317
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1530s 2s/step - accuracy: 0.4364 - loss: 1.6074 - val_accuracy: 0.4852 - val_loss: 1.4505
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1556s 2s/step - accuracy: 0.4898 - loss: 1.4644 - val_accuracy: 0.5086 - val_loss: 1.3887
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1539s 2s/step - accuracy: 0.5220 - loss: 1.3803 - val_accuracy: 0.5220 - val_loss: 1.3418
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1555s 2s/step - accuracy: 0.5428 - loss: 1.3148 - val_accuracy: 0.5504 - val_loss: 1.2831
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1548s 2s/step - accuracy: 0.5635 - loss: 1.2634 - val_accuracy: 0.5660 - val_loss: 1.2214
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1537s 2s/step - accuracy: 0.5794 - loss: 1.2215 - val_accuracy: 0.5340 - val_loss: 1.3093
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 1554s 2s/step - accuracy: 0.5947 - loss: 1.1750 - 